# IEC 60870-5-104 — gardu dan RTU

Master SCADA dan RTU bay penyulang 20 kV simulasi berbicara lewat link TCP dalam memori. Ganti `UseInMemory(rtuListener)` dengan `UseTcp("10.0.0.5")` untuk menjangkau RTU sungguhan — perintah hanya pada peralatan yang Anda berwenang operasikan.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.18.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.18.0-preview.1"

## APDU di jalur
Frame U memulai dan menguji link, frame S mengakui, frame I membawa ASDU bernomor.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Protocols.Iec104;
using IoTCom.Net.Transports;

Console.WriteLine(Iec104Apdu.Parse(Convert.FromHexString("680407000000")));
Console.WriteLine(Iec104Apdu.Parse(Convert.FromHexString("680401000400")));
var giFrame = Convert.FromHexString("680E0000000064010600010000000014");
Console.WriteLine(Iec104Apdu.Parse(giFrame));
foreach (var f in Iec104Apdu.Describe(giFrame)) Console.WriteLine($"{f.Name,-8} {f.Value}");

## Hubungkan dan interogasi
STARTDT, lalu interogasi umum mengembalikan semua nilai; counter datang lewat interogasi counter.

In [ ]:
var rtuListener = new InMemoryTransportListener("notebook-rtu");
var rtu = Iec104SubstationSimulator.Create(o => { o.ListenInMemory(rtuListener); o.RequireSelectBeforeOperate = true; });
await rtu.StartAsync();
var scada = Iec104Client.Create(o => o.UseInMemory(rtuListener).AllowCommands());
await scada.ConnectAsync();
foreach (var p in (await scada.InterrogateAsync()).OrderBy(p => p.Object.Address)) Console.WriteLine(p);
Console.WriteLine((await scada.CounterInterrogateAsync()).Single());

## Perubahan spontan bertanda waktu
Setelah interogasi RTU hanya mengirim perubahan, masing-masing dengan tanda waktu CP56Time2a.

In [ ]:
var changes = new System.Collections.Concurrent.ConcurrentQueue<Iec104PointValue>();
scada.PointReceived += p => { if (p.Cause == Iec104Cause.Spontaneous) changes.Enqueue(p); };
rtu.Trip();
await Task.Delay(300);
foreach (var c in changes.Where(c => c.Object.Address < 2000)) Console.WriteLine(c);

## Select-before-operate dan interlock
Pemutus perlu select lalu execute; RTU menolak menutup selama trip proteksi masih terkunci.

In [ ]:
try { await scada.DoubleCommandAsync(Iec104SubstationSimulator.Ioa.BreakerCommand, on: true, selectBeforeOperate: true); }
catch (IoTCom.Net.DeviceException e) { Console.WriteLine("refused: " + e.Message); }
await scada.SingleCommandAsync(Iec104SubstationSimulator.Ioa.TripReset, true, selectBeforeOperate: true);
var term = await scada.DoubleCommandAsync(Iec104SubstationSimulator.Ioa.BreakerCommand, on: true, selectBeforeOperate: true);
Console.WriteLine(term);
await Task.Delay(200);
Console.WriteLine(scada.Points[(1, Iec104SubstationSimulator.Ioa.Breaker)]);
await scada.DisposeAsync();
await rtu.DisposeAsync();

## Lebih lanjut
`iotcom iec104 gi --sim`, `iotcom iec104 monitor --sim`, `iotcom iec104 command double 5001 off --sbo --sim --allow-write`, `iotcom iec104 serve`, *Kendali gardu lewat IEC 104* di Gallery, dan sampel Iec104Scada. Lihat `docs/id/protocols/iec104.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*